<a href="https://colab.research.google.com/github/mmuputisi/Adv-Py_Data_Analysis/blob/main/ESG%20change%20model%20Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
"""
Standalone program for ESG-Tobin's Q analysis using Change in Lagged ESG models.
Properly aligned with t-1 ESG scores vs t financial performance.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from linearmodels.panel import PanelOLS
import warnings
warnings.filterwarnings('ignore')

class ChangeESGAnalysis:
    """
    Standalone class for running Change in Lagged ESG models on Tobin's Q.
    Properly aligned: ESG changes from t-2 to t-1 predict Tobin's Q at t.
    """

    def __init__(self, data_path, output_dir):
        """
        Initialize the Change in ESG analysis.

        Parameters:
        -----------
        data_path : str
            Path to the Excel data file
        output_dir : str
            Directory to save output files
        """
        self.data_path = data_path
        self.output_dir = output_dir
        self.df = None
        self.change_data = {}
        self.results = {}
        self.results_summary = []  # For tabular export

        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Setup plotting style
        plt.style.use('seaborn-v0_8-whitegrid')
        sns.set_palette("husl")

    def load_and_prepare_data(self):
        """
        Load data and prepare basic transformations.
        """
        print("=" * 70)
        print("LOADING AND PREPARING DATA FOR CHANGE IN ESG ANALYSIS")
        print("=" * 70)

        try:
            # Load data
            self.df = pd.read_excel(self.data_path)
            print(f"✓ Loaded data from: {self.data_path}")

            # Set multi-index
            if 'Firm' in self.df.columns and 'Year' in self.df.columns:
                self.df = self.df.set_index(['Firm', 'Year']).sort_index()
                print("✓ Set multi-index (Firm, Year)")
            else:
                print("⚠ Warning: 'Firm' and/or 'Year' columns not found")
                print(f"  Available columns: {list(self.df.columns)}")
                self.df = self.df.set_index([self.df.columns[0], self.df.columns[1]]).sort_index()
                print(f"  Using {self.df.index.names} as index")

            # Rename columns if needed
            column_renames = {}
            if "Tobin's Q" in self.df.columns:
                column_renames["Tobin's Q"] = "Tobin_Q"

            if column_renames:
                self.df = self.df.rename(columns=column_renames)
                print(f"✓ Renamed columns: {column_renames}")

            # Create basic control variables
            if 'Total Assets' in self.df.columns:
                self.df['Size'] = np.log(self.df['Total Assets'].replace(0, np.nan))
                print("✓ Created Size variable (log of Total Assets)")
            else:
                print("⚠ Warning: 'Total Assets' column not found")

            if 'Total Liabilities' in self.df.columns and 'Total Assets' in self.df.columns:
                self.df['Leverage'] = self.df['Total Liabilities'] / self.df['Total Assets'].replace(0, np.nan)
                print("✓ Created Leverage variable")
            else:
                print("⚠ Warning: Could not create Leverage variable")

            # Check for ESG columns
            esg_columns = [col for col in ['E', 'S', 'G'] if col in self.df.columns]
            if len(esg_columns) == 3:
                print(f"✓ Found ESG columns: {esg_columns}")
            else:
                print(f"⚠ Missing ESG columns. Found: {esg_columns}")

            print(f"\n✓ Loaded {len(self.df)} observations")
            print(f"✓ Number of firms: {self.df.index.get_level_values(0).nunique()}")
            print(f"✓ Years: {sorted(self.df.index.get_level_values(1).unique())}")

            return self.df

        except Exception as e:
            print(f"✗ Error loading data: {e}")
            import traceback
            traceback.print_exc()
            return None

    def create_sector_variable(self):
        """
        Create Consumer Staples sector variable.
        """
        print("\n" + "-" * 50)
        print("CREATING SECTOR VARIABLES")
        print("-" * 50)

        # Define Consumer Staples firms
        consumer_staples_firms = [
            'Associated British Foods', 'British American Tobacco', 'Coca Cola HBC AG',
            'Diageo', 'J. Sainsbury', 'Reckitt Benckiser', 'Tesco', 'Unilever', 'Haleon Plc'
        ]

        # Get unique firm names
        firm_names = self.df.index.get_level_values(0).unique()
        print(f"Found {len(firm_names)} unique firms")

        # Check which Consumer Staples firms are in the data
        found_cs_firms = [firm for firm in consumer_staples_firms if firm in firm_names]
        print(f"Found {len(found_cs_firms)} Consumer Staples firms in data: {found_cs_firms}")

        # Create sector variable
        sectors = []
        consumer_staples_dummy = []

        for firm in self.df.index.get_level_values(0):
            if firm in consumer_staples_firms:
                sectors.append('Consumer Staples')
                consumer_staples_dummy.append(1)
            else:
                sectors.append('Other')
                consumer_staples_dummy.append(0)

        self.df['Sector'] = sectors
        self.df['ConsumerStaples'] = consumer_staples_dummy

        print(f"\n✓ Created sector variables:")
        print(f"  Consumer Staples firms: {self.df[self.df['ConsumerStaples']==1].index.get_level_values(0).nunique()}")
        print(f"  Other sectors firms: {self.df[self.df['ConsumerStaples']==0].index.get_level_values(0).nunique()}")
        print(f"  Consumer Staples observations: {self.df['ConsumerStaples'].sum()}")

        return self.df

    def create_aligned_change_variables(self, max_lag=5):
        """
        Create properly aligned change variables.
        For change over k years: ΔkESG(t-1) = ESG(t-1) - ESG(t-1-k)
        This change predicts Tobin's Q at time t.

        Parameters:
        -----------
        max_lag : int
            Maximum lag period to create
        """
        print(f"\n{'='*70}")
        print(f"CREATING ALIGNED CHANGE VARIABLES (t-1 ESG predicting t Tobin's Q)")
        print(f"Maximum lag period: {max_lag} years")
        print(f"{'='*70}")

        # Check which ESG components are available
        esg_components = [c for c in ['E', 'S', 'G'] if c in self.df.columns]
        print(f"Available ESG components: {esg_components}")

        if not esg_components:
            print("⚠ No ESG components found!")
            return self.df

        # Create lagged variables for all periods
        for lag in range(1, max_lag + 1):
            for component in esg_components:
                # Lagged ESG at t-1
                lag1_col = f'{component}_lag1'
                if lag1_col not in self.df.columns:
                    self.df[lag1_col] = self.df.groupby(level=0)[component].shift(1)

                # Lagged ESG at t-1-k (for k-year change)
                lag_kplus1_col = f'{component}_lag{lag+1}'
                if lag_kplus1_col not in self.df.columns and lag > 1:
                    self.df[lag_kplus1_col] = self.df.groupby(level=0)[component].shift(lag+1)

                # Also create simple lag k for level effects
                lag_k_col = f'{component}_lag{lag}'
                if lag_k_col not in self.df.columns:
                    self.df[lag_k_col] = self.df.groupby(level=0)[component].shift(lag)

        # Create change variables
        for lag in range(1, max_lag + 1):
            print(f"\nCreating {lag}-year change variables (Δ{lag}ESG(t-1) = ESG(t-1) - ESG(t-{lag+1})):")

            for component in esg_components:
                # Current ESG at t-1
                current = f'{component}_lag1'
                # Past ESG at t-1-lag
                past = f'{component}_lag{lag+1}'

                if current in self.df.columns and past in self.df.columns:
                    # Absolute change
                    change_col = f'{component}_chg{lag}_aligned'
                    self.df[change_col] = self.df[current] - self.df[past]

                    # Store metadata
                    self.change_data[change_col] = {
                        'component': component,
                        'lag_period': lag,
                        'type': 'absolute',
                        'formula': f'{component}(t-1) - {component}(t-{lag+1})'
                    }

                    non_null = self.df[change_col].notna().sum()
                    mean_val = self.df[change_col].mean()
                    print(f"  ✓ {change_col}: {non_null} non-null, mean={mean_val:.4f}")

                    # Percentage change (avoid division by zero)
                    pct_col = f'{component}_pctchg{lag}_aligned'
                    denominator = self.df[past].replace(0, np.nan)
                    self.df[pct_col] = (self.df[current] - self.df[past]) / denominator

                    self.change_data[pct_col] = {
                        'component': component,
                        'lag_period': lag,
                        'type': 'percentage',
                        'formula': f'[{component}(t-1) - {component}(t-{lag+1})] / {component}(t-{lag+1})'
                    }

                    non_null = self.df[pct_col].notna().sum()
                    mean_val = self.df[pct_col].mean()
                    print(f"  ✓ {pct_col}: {non_null} non-null, mean={mean_val:.4%}")

        # Summary
        change_cols = [col for col in self.df.columns if '_chg' in col or '_pctchg' in col]
        print(f"\n✓ Total aligned change variables created: {len(change_cols)}")

        return self.df

    def run_aligned_model(self, change_period=1, change_type='absolute',
                         y_var='Tobin_Q', sector='All', include_levels=True):
        """
        Run a properly aligned change in ESG model.
        Tobin's Q at time t is predicted by ESG changes from t-1-lag to t-1.

        Parameters:
        -----------
        change_period : int
            Period over which change is calculated (k years)
        change_type : str
            Type of change ('absolute' or 'percentage')
        y_var : str
            Dependent variable (Tobin's Q at time t)
        sector : str
            Sector to analyze ('All' or 'CS')
        include_levels : bool
            Whether to include lagged ESG levels (at t-1)
        """
        print(f"\n{'='*70}")
        print(f"ALIGNED CHANGE IN ESG MODEL")
        print(f"Change period: {change_period} year(s) [Δ from t-{change_period+1} to t-1]")
        print(f"Type: {change_type}, Sector: {sector}, Y: {y_var}, Include levels: {include_levels}")
        print(f"{'='*70}")

        # Filter data by sector
        if sector == 'CS':
            if 'ConsumerStaples' not in self.df.columns:
                self.create_sector_variable()
            df_filtered = self.df[self.df['ConsumerStaples'] == 1].copy()
        else:
            df_filtered = self.df.copy()
            if 'ConsumerStaples' not in self.df.columns:
                self.create_sector_variable()

        # Define change variable suffix
        suffix = 'chg' if change_type == 'absolute' else 'pctchg'
        aligned_suffix = f'{suffix}{change_period}_aligned'

        # Define model variables
        model_vars = []

        # Add change variables
        change_vars = []
        for component in ['E', 'S', 'G']:
            change_var = f'{component}_{aligned_suffix}'
            if change_var in df_filtered.columns:
                change_vars.append(change_var)
                model_vars.append(change_var)
            else:
                print(f"⚠ Change variable {change_var} not found")

        # Add lagged ESG levels (at t-1) if requested
        level_vars = []
        if include_levels:
            for component in ['E', 'S', 'G']:
                level_var = f'{component}_lag1'
                if level_var in df_filtered.columns:
                    level_vars.append(level_var)
                    model_vars.append(level_var)

        # Add control variables
        control_vars = []
        for var in ['ROA', 'Size', 'Leverage']:
            if var in df_filtered.columns:
                control_vars.append(var)
                model_vars.append(var)

        # Add sector dummy if analyzing all firms
        if sector == 'All':
            model_vars.append('ConsumerStaples')

        # Clean data - use y_var at time t, predictors at t-1 and changes
        all_vars = model_vars + [y_var]
        df_clean = df_filtered.dropna(subset=all_vars)

        if len(df_clean) < 10:
            print(f"✗ Insufficient data: Only {len(df_clean)} observations")
            return None

        # Print data summary
        print(f"\nData Summary:")
        print(f"  Observations: {len(df_clean)}")
        print(f"  Firms: {df_clean.index.get_level_values(0).nunique()}")
        print(f"  Years: {sorted(df_clean.index.get_level_values(1).unique())}")

        # Prepare data for regression
        y = df_clean[y_var]
        X = df_clean[model_vars]
        X = pd.concat([pd.Series(1, index=X.index, name='const'), X], axis=1)

        print(f"\nRegression variables:")
        print(f"  Y: {y_var} (time t)")
        print(f"  Change vars: {change_vars}")
        if include_levels:
            print(f"  Level vars (t-1): {level_vars}")
        print(f"  Controls: {control_vars}")

        try:
            # Run regression
            model = PanelOLS(y, X, entity_effects=False, time_effects=False)
            results = model.fit(cov_type='robust')

            # Store results
            model_key = f'Change_{change_period}yr_{change_type}_levels{include_levels}_{sector}_{y_var}'
            self.results[model_key] = {
                'results': results,
                'model_name': f'Change in ESG ({change_period}yr, {change_type})',
                'change_period': change_period,
                'change_type': change_type,
                'include_levels': include_levels,
                'y_var': y_var,
                'sector': sector,
                'nobs': results.nobs,
                'r2': results.rsquared,
                'change_vars': change_vars,
                'level_vars': level_vars if include_levels else [],
                'control_vars': control_vars
            }

            # Add to summary table with fixed column order
            summary_row = {
                'Model': f'{change_period}yr_{change_type}_{sector}',
                'Period': change_period,
                'Type': change_type,
                'Sector': sector,
                'Levels': include_levels,
                'Constant': results.params.get('const', np.nan),
                'E': results.params.get(f'E_{aligned_suffix}', np.nan),
                'S': results.params.get(f'S_{aligned_suffix}', np.nan),
                'G': results.params.get(f'G_{aligned_suffix}', np.nan),
                'E_lag1': results.params.get('E_lag1', np.nan) if include_levels else np.nan,
                'S_lag1': results.params.get('S_lag1', np.nan) if include_levels else np.nan,
                'G_lag1': results.params.get('G_lag1', np.nan) if include_levels else np.nan,
                'ROA': results.params.get('ROA', np.nan),
                'Size': results.params.get('Size', np.nan),
                'Leverage': results.params.get('Leverage', np.nan),
                'ConsumerStaples': results.params.get('ConsumerStaples', np.nan) if sector == 'All' else np.nan,
                'R2': results.rsquared,
                'N': results.nobs,
                'E_pval': results.pvalues.get(f'E_{aligned_suffix}', np.nan),
                'S_pval': results.pvalues.get(f'S_{aligned_suffix}', np.nan),
                'G_pval': results.pvalues.get(f'G_{aligned_suffix}', np.nan)
            }
            self.results_summary.append(summary_row)

            # Print results
            print(f"\n✓ Regression successful!")
            print(f"  R-squared: {results.rsquared:.4f}")
            print(f"  Observations: {results.nobs}")

            # Print change coefficients
            print(f"\nChange in ESG Coefficients ({change_type} change from t-{change_period+1} to t-1):")
            for component in ['E', 'S', 'G']:
                change_var = f'{component}_{aligned_suffix}'
                if change_var in results.params:
                    coeff = results.params[change_var]
                    pval = results.pvalues.get(change_var, 1.0)
                    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
                    print(f"  Δ{component}: {coeff:.4f}{sig} (p={pval:.4f})")

            # Print level coefficients if included
            if include_levels:
                print(f"\nLagged ESG Levels (t-1):")
                for component in ['E', 'S', 'G']:
                    level_var = f'{component}_lag1'
                    if level_var in results.params:
                        coeff = results.params[level_var]
                        pval = results.pvalues.get(level_var, 1.0)
                        sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
                        print(f"  {component}(t-1): {coeff:.4f}{sig} (p={pval:.4f})")

            # Print control variables
            print(f"\nControl Variables:")
            for var in control_vars:
                if var in results.params:
                    coeff = results.params[var]
                    pval = results.pvalues.get(var, 1.0)
                    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
                    print(f"  {var}: {coeff:.4f}{sig} (p={pval:.4f})")

            return results

        except Exception as e:
            print(f"\n✗ Error running regression: {e}")
            import traceback
            traceback.print_exc()
            return None

    def run_comprehensive_analysis(self, periods=None, change_types=None,
                                  y_var='Tobin_Q', sectors=None):
        """
        Run comprehensive change analysis for multiple periods.

        Parameters:
        -----------
        periods : list
            List of change periods to analyze (1-5 years)
        change_types : list
            List of change types ['absolute', 'percentage']
        y_var : str
            Dependent variable
        sectors : list
            List of sectors ['All', 'CS']
        """
        if periods is None:
            periods = [1, 2, 3, 4, 5]

        if change_types is None:
            change_types = ['absolute', 'percentage']

        if sectors is None:
            sectors = ['All', 'CS']

        print("=" * 70)
        print("COMPREHENSIVE ALIGNED CHANGE IN ESG ANALYSIS")
        print("=" * 70)
        print(f"Analyzing change periods: {periods} years")
        print(f"Change types: {change_types}")
        print(f"Sectors: {sectors}")
        print(f"Dependent variable: {y_var}")

        # Create all aligned change variables first
        self.create_aligned_change_variables(max_lag=max(periods))

        # Run models for each combination
        print("\n" + "=" * 70)
        print("RUNNING REGRESSION MODELS")
        print("=" * 70)

        for period in periods:
            for change_type in change_types:
                for sector in sectors:
                    for include_levels in [False, True]:
                        self.run_aligned_model(
                            change_period=period,
                            change_type=change_type,
                            y_var=y_var,
                            sector=sector,
                            include_levels=include_levels
                        )

        # Create and save summary tables
        self.create_summary_tables()

    def create_summary_tables(self):
        """
        Create formatted summary tables with consistent column ordering.
        """
        if not self.results_summary:
            print("No results to summarize")
            return

        # Convert to DataFrame
        summary_df = pd.DataFrame(self.results_summary)

        # Define column order for main coefficients table
        main_cols = ['Model', 'Period', 'Type', 'Sector', 'Levels',
                     'Constant', 'E', 'S', 'G',
                     'E_lag1', 'S_lag1', 'G_lag1',
                     'ROA', 'Size', 'Leverage', 'ConsumerStaples',
                     'R2', 'N']

        # Filter to available columns
        available_main_cols = [col for col in main_cols if col in summary_df.columns]
        main_table = summary_df[available_main_cols].copy()

        # Round numeric columns
        numeric_cols = ['Constant', 'E', 'S', 'G', 'E_lag1', 'S_lag1', 'G_lag1',
                       'ROA', 'Size', 'Leverage', 'ConsumerStaples', 'R2']
        for col in numeric_cols:
            if col in main_table.columns:
                main_table[col] = main_table[col].round(4)

        # Save main table
        main_path = os.path.join(self.output_dir, 'change_esg_coefficients.xlsx')
        main_table.to_excel(main_path, index=False)
        print(f"\n✓ Saved coefficients table to: {main_path}")

        # Create significance table
        sig_cols = ['Model', 'Period', 'Type', 'Sector', 'Levels',
                   'E_pval', 'S_pval', 'G_pval', 'R2', 'N']
        available_sig_cols = [col for col in sig_cols if col in summary_df.columns]
        sig_table = summary_df[available_sig_cols].copy()

        # Add significance stars
        for comp in ['E', 'S', 'G']:
            pval_col = f'{comp}_pval'
            star_col = f'{comp}_sig'
            if pval_col in sig_table.columns:
                sig_table[star_col] = sig_table[pval_col].apply(
                    lambda x: '***' if x < 0.01 else '**' if x < 0.05 else '*' if x < 0.10 else ''
                )

        # Round p-values
        for col in ['E_pval', 'S_pval', 'G_pval']:
            if col in sig_table.columns:
                sig_table[col] = sig_table[col].round(4)

        # Save significance table
        sig_path = os.path.join(self.output_dir, 'change_esg_significance.xlsx')
        sig_table.to_excel(sig_path, index=False)
        print(f"✓ Saved significance table to: {sig_path}")

        # Create separate sheets for different sectors and level specifications
        with pd.ExcelWriter(os.path.join(self.output_dir, 'change_esg_detailed.xlsx')) as writer:
            # All models
            main_table.to_excel(writer, sheet_name='All_Models', index=False)

            # By sector and levels
            for sector in ['All', 'CS']:
                for levels in [False, True]:
                    subset = main_table[(main_table['Sector'] == sector) &
                                       (main_table['Levels'] == levels)]
                    if not subset.empty:
                        sheet_name = f'{sector}_Levels{levels}'
                        subset.to_excel(writer, sheet_name=sheet_name, index=False)

        print(f"✓ Saved detailed tables to: {os.path.join(self.output_dir, 'change_esg_detailed.xlsx')}")

        # Display best models
        self.display_best_models(main_table)

    def display_best_models(self, main_table):
        """
        Display the best performing models.
        """
        print("\n" + "=" * 70)
        print("BEST MODELS BY R-SQUARED")
        print("=" * 70)

        # Overall best
        best_idx = main_table['R2'].idxmax()
        best = main_table.loc[best_idx]
        print("\n🏆 OVERALL BEST MODEL:")
        print(f"  Model: {best['Model']}")
        print(f"  R²: {best['R2']:.4f}")
        print(f"  N: {best['N']}")
        print(f"  Coefficients:")
        print(f"    E: {best['E']:.4f}")
        print(f"    S: {best['S']:.4f}")
        print(f"    G: {best['G']:.4f}")
        if 'E_lag1' in best and pd.notna(best['E_lag1']):
            print(f"    E(t-1): {best['E_lag1']:.4f}")
            print(f"    S(t-1): {best['S_lag1']:.4f}")
            print(f"    G(t-1): {best['G_lag1']:.4f}")
        print(f"    ROA: {best['ROA']:.4f}")
        print(f"    Size: {best['Size']:.4f}")
        print(f"    Leverage: {best['Leverage']:.4f}")

        # Best by sector
        for sector in ['All', 'CS']:
            sector_models = main_table[main_table['Sector'] == sector]
            if not sector_models.empty:
                best_sector_idx = sector_models['R2'].idxmax()
                best_sector = sector_models.loc[best_sector_idx]
                print(f"\n📊 BEST {sector} SECTOR MODEL:")
                print(f"  Model: {best_sector['Model']}")
                print(f"  R²: {best_sector['R2']:.4f}")
                print(f"  S coefficient: {best_sector['S']:.4f}")

        # Best by change period
        print("\n📈 BEST BY CHANGE PERIOD:")
        for period in sorted(main_table['Period'].unique()):
            period_models = main_table[main_table['Period'] == period]
            if not period_models.empty:
                best_period_idx = period_models['R2'].idxmax()
                best_period = period_models.loc[best_period_idx]
                print(f"  {period}-year: R²={best_period['R2']:.4f} ({best_period['Model']})")

    def save_results(self):
        """
        Save all results to Excel files.
        """
        print("\n" + "=" * 70)
        print("SAVING RESULTS")
        print("=" * 70)

        # Save change variables summary
        if self.change_data:
            change_summary = []
            for col_name, data_info in self.change_data.items():
                if col_name in self.df.columns:
                    non_null = self.df[col_name].notna().sum()
                    mean_val = self.df[col_name].mean()
                    std_val = self.df[col_name].std()

                    change_summary.append({
                        'Variable': col_name,
                        'Component': data_info['component'],
                        'Lag_Period': data_info['lag_period'],
                        'Type': data_info['type'],
                        'Formula': data_info['formula'],
                        'Non_Null': non_null,
                        'Mean': mean_val,
                        'Std': std_val
                    })

            if change_summary:
                change_df = pd.DataFrame(change_summary)
                change_path = os.path.join(self.output_dir, 'aligned_change_variables.xlsx')
                change_df.to_excel(change_path, index=False)
                print(f"✓ Saved aligned change variables to: {change_path}")

        print(f"\n✓ All results saved to: {self.output_dir}")


def main():
    """
    Main function to run the aligned Change in ESG analysis.
    """
    # Configuration
    DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/FTSE Data/FTSE data.xlsx'
    OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/FTSE Data/aligned_change_output'

    print("=" * 70)
    print("ALIGNED CHANGE IN LAGGED ESG ANALYSIS")
    print("=" * 70)
    print("Proper timing: ESG changes from t-1-k to t-1 predict Tobin's Q at t")

    # Initialize analysis
    analyzer = ChangeESGAnalysis(DATA_PATH, OUTPUT_DIR)

    # Step 1: Load and prepare data
    print("\n" + "=" * 70)
    print("STEP 1: LOADING DATA")
    print("=" * 70)

    df = analyzer.load_and_prepare_data()

    if df is None:
        print("✗ Failed to load data. Exiting.")
        return

    # Step 2: Create sector variable
    print("\n" + "=" * 70)
    print("STEP 2: CREATING SECTOR VARIABLES")
    print("=" * 70)

    analyzer.create_sector_variable()

    # Step 3: Run comprehensive analysis for 1-5 years
    print("\n" + "=" * 70)
    print("STEP 3: COMPREHENSIVE ALIGNED ANALYSIS")
    print("=" * 70)

    analyzer.run_comprehensive_analysis(
        periods=[1, 2, 3, 4, 5],
        change_types=['absolute'],
        y_var='Tobin_Q',
        sectors=['All', 'CS']
    )

    # Step 4: Save all results
    print("\n" + "=" * 70)
    print("STEP 4: SAVING FINAL RESULTS")
    print("=" * 70)

    analyzer.save_results()

    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETE")
    print("=" * 70)


if __name__ == "__main__":
    main()

ALIGNED CHANGE IN LAGGED ESG ANALYSIS
Proper timing: ESG changes from t-1-k to t-1 predict Tobin's Q at t

STEP 1: LOADING DATA
LOADING AND PREPARING DATA FOR CHANGE IN ESG ANALYSIS
✓ Loaded data from: /content/drive/MyDrive/Colab Notebooks/FTSE Data/FTSE data.xlsx
✓ Set multi-index (Firm, Year)
✓ Created Size variable (log of Total Assets)
✓ Created Leverage variable
✓ Found ESG columns: ['E', 'S', 'G']

✓ Loaded 388 observations
✓ Number of firms: 40
✓ Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

STEP 2: CREATING SECTOR VARIABLES

--------------------------------------------------
CREATING SECTOR VARIABLES
--------------------------------------------------
Found 40 unique firms
Found 8 Consumer Staples firms in data: ['Associated British Foods', 'British American Tobacco', 'Coca Cola HBC AG', 'Diageo', 'J. Sainsbury', 'Reckitt Benckiser', 'Tesco', 'Unilever']

✓ Created sector variables:
  Consumer Staples firms: 8
  Other sectors firms: 32
  Consumer Staples 